In [ ]:
%%capture
%run ../part_2/part2.ipynb
import math
import numpy as np
import pandas as pd
from collections import defaultdict

In [ ]:
#Does the same as the function in part2 search, but now we divide the ranking in an other function, since we need all the retrived documents to do the score of our engine.
def get_candidates(query):
    """
    Conjunctive (AND) retrieval over the per-field inverted indexes.

    Returns
    -------
    q_terms   : list of unique pre-processed query terms
    docs      : set of doc_ids that contain ALL the query terms (in any field)
    postings  : dict (field, term) -> {doc_id: term frequency in that field}
    term_docs : dict term -> set of doc_ids containing the term in any field
    """
    q_terms = list(dict.fromkeys(build_terms(query)))   # unique, keeps order
    if not q_terms:
        return [], set(), {}, {}

    postings, term_docs = {}, {t: set() for t in q_terms}
    for t in q_terms:
        for field in index:
            tf = {}
            for doc_id, positions in index[field].get(t, []):
                tf[doc_id] = len(positions)
                term_docs[t].add(doc_id)
            postings[(field, t)] = tf

    docs = set.intersection(*term_docs.values())
    return q_terms, docs, postings, term_docs


def top_k_from_scores(scores, top_k):
    """Sort a {doc_id: score} dict (desc) and cut it to top_k. Ties are broken by doc_id (deterministic)."""
    ranked = sorted(scores.items(), key=lambda x: (-x[1], x[0]))
    return ranked[:top_k]

In [ ]:
def build_tfidf_structures():
    """Pre-computes idf, the weighted TF-IDF vector of every document (sparse dict) and its L2 norm."""
    vocab_docs = defaultdict(set)
    for field in index:
        for term, plist in index[field].items():
            for doc_id, _ in plist:
                vocab_docs[term].add(doc_id)
    idf = {t: math.log(N / len(d)) for t, d in vocab_docs.items()}

    doc_vec = [defaultdict(float) for _ in range(N)]
    for field, idx in index.items():
        w = weights[field]
        if w == 0:
            continue
        for term, plist in idx.items():
            for doc_id, positions in plist:
                doc_vec[doc_id][term] += w * (1 + math.log(len(positions))) * idf[term]

    norms = np.array([math.sqrt(sum(v * v for v in d.values())) for d in doc_vec])
    return idf, doc_vec, norms

idf_tfidf, doc_vec_tfidf, doc_norm_tfidf = build_tfidf_structures()


def rank_tfidf_cosine(query, top_k=20):
    """Conjunctive search ranked by TF-IDF + cosine similarity."""
    q_terms, docs, _, _ = get_candidates(query)
    if not docs:
        return []
    q_tf = {t: q_terms.count(t) for t in q_terms}
    q_w = {t: (1 + math.log(q_tf[t])) * idf_tfidf.get(t, 0.0) for t in q_terms}
    q_norm = math.sqrt(sum(v * v for v in q_w.values()))
    if q_norm == 0:                      # every term appears in every document -> idf = 0
        return top_k_from_scores({d: 0.0 for d in docs}, top_k)

    scores = {}
    for d in docs:
        if doc_norm_tfidf[d] == 0:
            scores[d] = 0.0
            continue
        dot = sum(q_w[t] * doc_vec_tfidf[d].get(t, 0.0) for t in q_terms)
        scores[d] = dot / (q_norm * doc_norm_tfidf[d])
    return top_k_from_scores(scores, top_k)

In [ ]:
field_len = {f: np.array([len(t) for t in field_terms[f]], dtype=float) for f in field_terms}
avg_len = {f: max(field_len[f].mean(), 1e-9) for f in field_len}

K1, B = 1.2, 0.75


def bm25_scores(query, k1=K1, b=B):
    """Returns (docs, {doc_id: BM25 score}) for the conjunctive candidates of the query."""
    q_terms, docs, postings, term_docs = get_candidates(query)
    scores = {}
    for d in docs:
        s = 0.0
        for t in q_terms:
            tf_tilde = 0.0
            for f in index:
                tf = postings[(f, t)].get(d, 0)
                if tf and weights[f] > 0:
                    tf_tilde += weights[f] * tf / (1 - b + b * field_len[f][d] / avg_len[f])
            n = len(term_docs[t])
            idf = math.log((N - n + 0.5) / (n + 0.5) + 1)
            s += idf * tf_tilde * (k1 + 1) / (tf_tilde + k1)
        scores[d] = s
    return scores


def rank_bm25(query, top_k=20, k1=K1, b=B):
    """Conjunctive search ranked by (field-weighted) BM25."""
    return top_k_from_scores(bm25_scores(query, k1, b), top_k)

In [ ]:
def to_number(series):
    """Parses strings like '2,999', '69% off' or '3.9' into floats (NaN if impossible)."""
    return pd.to_numeric(series.astype(str).str.replace(r"[^0-9.]", "", regex=True), errors="coerce")

rating_num   = to_number(df["average_rating"])
discount_num = to_number(df["discount"])
oos_flag     = df["out_of_stock"].astype(str).str.lower().isin(["true", "1"]).to_numpy()

rating_norm   = (rating_num.fillna(rating_num.median()) / 5.0).clip(0, 1).to_numpy()
discount_norm = (discount_num.fillna(discount_num.median()) / 100.0).clip(0, 1).to_numpy()

ALPHA, BETA, GAMMA = 0.2, 0.1, 0.6


def rank_custom(query, top_k=20, alpha=ALPHA, beta=BETA, gamma=GAMMA):
    """Conjunctive search ranked by BM25 boosted with rating, discount and stock availability."""
    bm25 = bm25_scores(query)
    if not bm25:
        return []
    max_bm25 = max(bm25.values()) or 1.0
    scores = {}
    for d, s in bm25.items():
        boost = 1 + alpha * rating_norm[d] + beta * discount_norm[d]
        penalty = gamma if oos_flag[d] else 1.0
        scores[d] = (s / max_bm25) * boost * penalty
    return top_k_from_scores(scores, top_k)

In [ ]:
def show_ranking(rank_fn, query, top_k=5):
    for rank, (d, s) in enumerate(rank_fn(query, top_k), start=1):
        print(f"{rank:>2}. {s:7.3f} | {pids[d]} | {df['title'].iat[d]} | {df['brand'].iat[d]} | rating={df['average_rating'].iat[d]}")

for name, fn in [("TF-IDF cosine", rank_tfidf_cosine), ("BM25", rank_bm25), ("Custom", rank_custom)]:
    print(f"\n=== {name} ===")
    show_ranking(fn, test_queries[1])

In [ ]:
def evaluate_ranker(rank_fn, k=10):
    """Same protocol as in Part 2: only documents with a label are kept, in ranking order."""
    all_y, rrs, rows = [], [], []
    for qid, q in validation_queries.items():
        ql = labels_df[labels_df["query_id"] == qid]
        rel = dict(zip(ql["pid"], ql["labels"]))
        ranked = rank_fn(q, top_k=len(pids))
        retrieved = [pids[d] for d, _ in ranked if pids[d] in rel][:k]
        y = [int(rel[p]) for p in retrieved]
        total = int((ql["labels"] == 1).sum())
        rr = rr_at_k(y, k)
        all_y.append(y); rrs.append(rr)
        rows.append({"query_id": qid, "P@10": precision_at_k(y, k), "R@10": recall_at_k(y, total, k),
                     "AP@10": avg_precision_at_k(y, k), "RR@10": rr, "NDCG@10": ndcg_at_k(y, k, n_relevant=total)})
    res = pd.DataFrame(rows).set_index("query_id")
    res.loc["mean"] = res.mean()
    res["MAP@10"] = map_at_k(all_y, k)[0]
    res["MRR@10"] = mrr_at_k(rrs)
    return res

for name, fn in [("TF-IDF cosine", rank_tfidf_cosine), ("BM25", rank_bm25), ("Custom", rank_custom)]:
    print(f"\n=== {name} ===")
    display(evaluate_ranker(fn).round(3))

In [ ]:
from gensim.models import Word2Vec

def doc_tokens(i):
    """All the tokens of document i that are used to represent it (seller excluded)."""
    return (field_terms["title"][i] + field_terms["description"][i] + field_terms["category"][i]
            + field_terms["brand"][i] + field_terms["details"][i])

corpus = [doc_tokens(i) for i in range(N)]

w2v = Word2Vec(sentences=corpus, vector_size=100, window=5, min_count=2, sg=1,
               epochs=20, workers=1, seed=42)      # workers=1 + seed -> reproducible

def text_vector(tokens):
    """Average of the word2vec vectors of the known tokens (zero vector if none is known)."""
    vecs = [w2v.wv[t] for t in tokens if t in w2v.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(w2v.vector_size)

doc_vecs = np.vstack([text_vector(t) for t in corpus])
doc_vec_norms = np.linalg.norm(doc_vecs, axis=1)

In [ ]:
def rank_word2vec(query, top_k=20):
    """Conjunctive search; candidates ranked by cosine(avg w2v(query), avg w2v(document))."""
    q_terms, docs, _, _ = get_candidates(query)
    if not docs:
        return []
    q_vec = text_vector(q_terms)
    q_norm = np.linalg.norm(q_vec)
    scores = {}
    for d in docs:
        denom = q_norm * doc_vec_norms[d]
        scores[d] = float(doc_vecs[d] @ q_vec / denom) if denom > 0 else 0.0
    return top_k_from_scores(scores, top_k)

w2v_rows = []
for qid, q in test_queries.items():
    for rank, (d, s) in enumerate(rank_word2vec(q, top_k=20), start=1):
        w2v_rows.append({"query_id": qid, "query": q, "rank": rank, "pid": pids[d],
                         "title": df["title"].iat[d], "w2v_cosine": round(s, 4)})
w2v_results = pd.DataFrame(w2v_rows)
display(w2v_results)

In [ ]:
# Same metrics on the validation queries, to compare with the other rankers
display(evaluate_ranker(rank_word2vec).round(3))